In [11]:
from datasets import load_dataset

dataset = load_dataset('smilegate-ai/kor_unsmile')
print(dataset)



DatasetDict({
    train: Dataset({
        features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels'],
        num_rows: 15005
    })
    valid: Dataset({
        features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels'],
        num_rows: 3737
    })
})


In [12]:
import pandas as pd

df = dataset['train'].to_pandas()
df.head()

,문장,여성/가족,남성,성소수자,인종/국적,연령,지역,종교,기타 혐오,악플/욕설,clean,개인지칭,labels
0,일안하는 시간은 쉬고싶어서 그런게 아닐까,0,0,0,0,0,0,0,0,0,1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
1,아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...,0,0,0,0,0,0,1,0,0,0,0,"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"
2,루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o doin 진짜 띵...,0,0,0,0,0,0,0,0,0,1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
3,홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...,0,0,0,0,0,0,0,0,0,1,0,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1]"
4,아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...,1,0,0,0,0,0,0,0,0,0,0,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]"


In [13]:
print(df['clean'].value_counts())
print(df['clean'].value_counts(normalize=True) * 100)

clean
0    11266
1     3739
Name: count, dtype: int64
clean
0    75.081639
1    24.918361
Name: proportion, dtype: float64


In [14]:
# clean=1(정상) → label=0,  clean=0(비속어) → label=1
df['label'] = 1 - df['clean']

print(df[['문장', 'clean', 'label']].head())
print(df['label'].value_counts())

                                                  문장  clean  label
0                             일안하는 시간은 쉬고싶어서 그런게 아닐까      1      0
1  아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...      0      1
2  루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o  doin 진짜 띵...      1      0
3  홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...      1      0
4  아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...      0      1
label
1    11266
0     3739
Name: count, dtype: int64


In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('beomi/kcbert-base')
print("토크나이저 로드 완료")

토크나이저 로드 완료


In [16]:
test = "안녕하세요 반갑습니다"
result = tokenizer(test)

print("원문:", test)
print("토큰:", tokenizer.tokenize(test))
print("숫자:", result['input_ids'])

원문: 안녕하세요 반갑습니다
토큰: ['안녕', '##하세요', '반', '##갑', '##습니다']
숫자: [2, 19017, 8482, 1483, 4981, 8046, 3]


In [21]:
train_df = dataset['train'].to_pandas()
valid_df = dataset['valid'].to_pandas()

train_df['label'] = 1 - train_df['clean']
valid_df['label'] = 1 - valid_df['clean']

# 문장이랑 label만! (원본 labels 등 다른 컬럼 다 버림)
train_df = train_df[['문장', 'label']]
valid_df = valid_df[['문장', 'label']]

print(train_df.head())
print(train_df['label'].value_counts())

                                                  문장  label
0                             일안하는 시간은 쉬고싶어서 그런게 아닐까      0
1  아동성범죄와 페도버는 기록바 끊어져 영원히 고통 받는다. 무슬림 50퍼 근친이다. ...      1
2  루나 솔로앨범 나왔을 때부터 머모 기운 있었음 ㅇㅇ Keep o  doin 진짜 띵...      0
3  홍팍에도 어버이연합인가 보내요 뭐 이런뎃글 있는데 이거 어버이연합측에 신고하면 그쪽...      0
4  아놔 왜 여기 댓들은 다 여자들이 김치녀라고 먼저 불렸다! 여자들은 더 심하게 그런...      1
label
1    11266
0     3739
Name: count, dtype: int64


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
valid_dataset = Dataset.from_pandas(valid_df)

print(train_dataset)

Dataset({
    features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels', 'label'],
    num_rows: 15005
})
Dataset({
    features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels', 'label'],
    num_rows: 3737
})


In [22]:
#토큰화 및 전제 적용

def tokenize_function(examples):
    return tokenizer(
        examples['문장'],
        padding='max_length',   # 길이를 일정하게 맞춘다
        truncation=True,         # 너무 길면 자른다
        max_length=128           # 최대 128 토큰
    )

#전체에 적용
train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)
print(train_tokenized)

Map: 100%|██████████| 3737/3737 [00:00<00:00, 28081.10 examples/s]

Dataset({
    features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 15005
})


In [23]:
columns_to_return = ['input_ids', 'token_type_ids', 'attention_mask', 'label']
train_tokenized.set_format(type='torch', columns=columns_to_return)
valid_tokenized.set_format(type='torch', columns=columns_to_return)

print(train_tokenized)
print("형식 설정 완료")

Dataset({
    features: ['문장', '여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설', 'clean', '개인지칭', 'labels', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 15005
})
형식 설정 완료


In [24]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    'beomi/kcbert-base',
    num_labels=2              #정상이면0, 비속어면1 -> 2개
)

print("모델 로드 완료")

c:\Users\User\profanity-cleaner\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--beomi--kcbert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11939.50it/s]
[transformers] BertForSequenceClassification

모델 로드 완료
